# Use Case 1: Ingestion - Scraping Data IDX ke Database

**Kelompok 1 (Extract)**: Scraping data dari API IDX (https://www.idx.co.id)

**Kelompok 2 (Load)**: Menyimpan data ke database SQLite

## Flow Eksekusi
1. **Extract** - Mengambil data stock summary dari IDX API menggunakan cloudscraper (bypass Cloudflare)
2. **Transform** - Pembersihan dan transformasi data (numeric conversion, string cleaning)
3. **Load** - Menyimpan ke database SQLite menggunakan SQLAlchemy ORM

## 1. Install & Import Library

In [1]:
!pip install curl_cffi pandas sqlalchemy

  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached greenlet-3.5.1-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached sqlalchemy-2.0.50-cp314-

In [2]:
import json
import time
from curl_cffi import requests as cf_requests
import pandas as pd
from sqlalchemy import Column, Integer, Float, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

## 2. Konfigurasi

In [3]:
IDX_API_URL = "https://www.idx.co.id/primary/TradingSummary/GetStockSummary"
IDX_API_PARAMS = {"length": 9999, "start": 0}
DB_PATH = "sqlite:///idx_stock.db"
REQUEST_TIMEOUT = 30

## 3. Definisi Model Database (SQLAlchemy ORM)

Tabel `stock_summary` menyimpan data ringkasan perdagangan saham harian dari IDX.

In [4]:
Base = declarative_base()


class StockSummary(Base):
    __tablename__ = "stock_summary"

    id = Column(Integer, primary_key=True, autoincrement=True)
    id_stock_summary = Column(Integer, nullable=False)
    date = Column(String, nullable=False)
    stock_code = Column(String, nullable=False, index=True)
    stock_name = Column(String, nullable=False)
    remarks = Column(String, nullable=True)
    previous = Column(Float, nullable=True)
    open_price = Column(Float, nullable=True)
    first_trade = Column(Float, nullable=True)
    high = Column(Float, nullable=True)
    low = Column(Float, nullable=True)
    close = Column(Float, nullable=True)
    change = Column(Float, nullable=True)
    volume = Column(Float, nullable=True)
    value = Column(Float, nullable=True)
    frequency = Column(Float, nullable=True)
    index_individual = Column(Float, nullable=True)
    offer = Column(Float, nullable=True)
    offer_volume = Column(Float, nullable=True)
    bid = Column(Float, nullable=True)
    bid_volume = Column(Float, nullable=True)
    listed_shares = Column(Float, nullable=True)
    tradeble_shares = Column(Float, nullable=True)
    weight_for_index = Column(Float, nullable=True)
    foreign_sell = Column(Float, nullable=True)
    foreign_buy = Column(Float, nullable=True)
    delisting_date = Column(String, nullable=True)
    non_regular_volume = Column(Float, nullable=True)
    non_regular_value = Column(Float, nullable=True)
    non_regular_frequency = Column(Float, nullable=True)

    def __repr__(self):
        return f"<StockSummary {self.stock_code} {self.date}>"


engine = create_engine(DB_PATH, echo=False)
Base.metadata.create_all(engine)
print(f"Database dibuat: {DB_PATH}")

Database dibuat: sqlite:///idx_stock.db


## 4. Extract - Scraping Data dari IDX API

Website IDX dilindungi oleh Cloudflare, sehingga kita menggunakan `cloudscraper` untuk bypass proteksi anti-bot.

In [5]:
print(f"[EXTRACT] Mengambil data dari {IDX_API_URL} ...")

# Metode 1: curl_cffi (TLS fingerprint impersonation - bypass Cloudflare)
try:
    resp = cf_requests.get(
        IDX_API_URL,
        params=IDX_API_PARAMS,
        impersonate="chrome",
        timeout=REQUEST_TIMEOUT,
    )
    print(f"[EXTRACT] HTTP Status: {resp.status_code}")
    if resp.status_code != 200:
        raise Exception(f"Status {resp.status_code}")
except Exception as e:
    print(f"[EXTRACT] curl_cffi gagal: {e}")
    print("[EXTRACT] Mencoba dengan metode alternatif ...")
    import cloudscraper
    scraper = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "windows", "desktop": True})
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://www.idx.co.id/",
        "Accept": "application/json, text/plain, */*",
    }
    import time
    for attempt in range(1, 4):
        resp = scraper.get(IDX_API_URL, params=IDX_API_PARAMS, timeout=REQUEST_TIMEOUT, headers=headers)
        print(f"[EXTRACT] Retry {attempt}/3 - Status: {resp.status_code}")
        if resp.status_code == 200:
            break
        time.sleep(3)
    if resp.status_code != 200:
        raise Exception(f"Gagal mengambil data setelah 3 percobaan. HTTP: {resp.status_code}")

payload = resp.json()
records = payload.get("data", [])
total = payload.get("recordsTotal", 0)
print(f"[EXTRACT] Berhasil mengambil {len(records)} record (total: {total})")
print(f"[EXTRACT] Contoh data pertama:")
import json
print(json.dumps(records[0], indent=2))

[EXTRACT] Mengambil data dari https://www.idx.co.id/primary/TradingSummary/GetStockSummary ...
[EXTRACT] HTTP Status: 200
[EXTRACT] Berhasil mengambil 959 record (total: 959)
[EXTRACT] Contoh data pertama:
{
  "No": 1,
  "IDStockSummary": 4024001,
  "Date": "2026-06-03T00:00:00",
  "StockCode": "AADI",
  "StockName": "Adaro Andalan Indonesia Tbk.",
  "Remarks": "CDMO1SD0F10000A121------------",
  "Previous": 8325.0,
  "OpenPrice": 8300.0,
  "FirstTrade": 8325.0,
  "High": 8425.0,
  "Low": 7850.0,
  "Close": 8000.0,
  "Change": -325.0,
  "Volume": 23469400.0,
  "Value": 190835972500.0,
  "Frequency": 10333.0,
  "IndexIndividual": 144.1,
  "Offer": 8025.0,
  "OfferVolume": 3400.0,
  "Bid": 8000.0,
  "BidVolume": 50500.0,
  "ListedShares": 7786891760.0,
  "TradebleShares": 7786891760.0,
  "WeightForIndex": 1486517637.0,
  "ForeignSell": 6189700.0,
  "ForeignBuy": 4475400.0,
  "DelistingDate": "",
  "NonRegularVolume": 280.0,
  "NonRegularValue": 2201261.0,
  "NonRegularFrequency": 4.0,
  

## 5. Transform - Pembersihan Data

- Hapus baris tanpa `stock_code` atau `date`
- Konversi kolom numerik
- Bersihkan string (strip, uppercase)

In [6]:
rows = []
for item in records:
    rows.append({
        "id_stock_summary": item.get("IDStockSummary"),
        "date": item.get("Date", "")[:10] if item.get("Date") else "",
        "stock_code": item.get("StockCode", ""),
        "stock_name": item.get("StockName", ""),
        "remarks": item.get("Remarks", ""),
        "previous": item.get("Previous"),
        "open_price": item.get("OpenPrice"),
        "first_trade": item.get("FirstTrade"),
        "high": item.get("High"),
        "low": item.get("Low"),
        "close": item.get("Close"),
        "change": item.get("Change"),
        "volume": item.get("Volume"),
        "value": item.get("Value"),
        "frequency": item.get("Frequency"),
        "index_individual": item.get("IndexIndividual"),
        "offer": item.get("Offer"),
        "offer_volume": item.get("OfferVolume"),
        "bid": item.get("Bid"),
        "bid_volume": item.get("BidVolume"),
        "listed_shares": item.get("ListedShares"),
        "tradeble_shares": item.get("TradebleShares"),
        "weight_for_index": item.get("WeightForIndex"),
        "foreign_sell": item.get("ForeignSell"),
        "foreign_buy": item.get("ForeignBuy"),
        "delisting_date": item.get("DelistingDate", ""),
        "non_regular_volume": item.get("NonRegularVolume"),
        "non_regular_value": item.get("NonRegularValue"),
        "non_regular_frequency": item.get("NonRegularFrequency"),
    })

df = pd.DataFrame(rows)
print(f"Data mentah: {len(df)} baris")
df.head()

Data mentah: 959 baris


,id_stock_summary,date,stock_code,stock_name,remarks,previous,open_price,first_trade,high,low,...,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency
0,4024001,2026-06-03,AADI,Adaro Andalan Indonesia Tbk.,CDMO1SD0F10000A121------------,8325.0,8300.0,8325.0,8425.0,7850.0,...,50500.0,7.786892e+09,7.786892e+09,1.486518e+09,6189700.0,4475400.0,,280.0,2201261.0,4.0
1,4024002,2026-06-03,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6825.0,6800.0,6825.0,6825.0,6525.0,...,96500.0,1.924688e+09,1.924688e+09,3.907117e+08,3581300.0,1990600.0,,0.0,0.0,0.0
2,4024003,2026-06-03,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,37.0,0.0,0.0,37.0,37.0,...,1100.0,3.935893e+09,3.935893e+09,1.298845e+09,0.0,0.0,,0.0,0.0,0.0
3,4024004,2026-06-03,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3500.0,0.0,0.0,3400.0,3070.0,...,100.0,6.208067e+08,6.208067e+08,8.132568e+07,1500.0,0.0,,0.0,0.0,0.0
4,4024005,2026-06-03,ABMM,ABM Investama Tbk.,XDMO1135000000A121------------,2390.0,2410.0,2410.0,2420.0,2260.0,...,100.0,2.753165e+09,2.753165e+09,4.138007e+08,411500.0,76500.0,,0.0,0.0,0.0


In [7]:
# Transform: pembersihan data
df = df.dropna(subset=["stock_code", "date"])

numeric_cols = [
    "previous", "open_price", "first_trade", "high", "low", "close",
    "change", "volume", "value", "frequency", "index_individual",
    "offer", "offer_volume", "bid", "bid_volume", "listed_shares",
    "tradeble_shares", "weight_for_index", "foreign_sell", "foreign_buy",
    "non_regular_volume", "non_regular_value", "non_regular_frequency",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["stock_name"] = df["stock_name"].astype(str).str.strip()
df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()

print(f"Data setelah pembersihan: {len(df)} baris")
df.head(10)

Data setelah pembersihan: 959 baris


,id_stock_summary,date,stock_code,stock_name,remarks,previous,open_price,first_trade,high,low,...,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency
0,4024001,2026-06-03,AADI,Adaro Andalan Indonesia Tbk.,CDMO1SD0F10000A121------------,8325.0,8300.0,8325.0,8425.0,7850.0,...,50500.0,7.786892e+09,7.786892e+09,1.486518e+09,6189700.0,4475400.0,,280.0,2201261.0,4.0
1,4024002,2026-06-03,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6825.0,6800.0,6825.0,6825.0,6525.0,...,96500.0,1.924688e+09,1.924688e+09,3.907117e+08,3581300.0,1990600.0,,0.0,0.0,0.0
2,4024003,2026-06-03,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,37.0,0.0,0.0,37.0,37.0,...,1100.0,3.935893e+09,3.935893e+09,1.298845e+09,0.0,0.0,,0.0,0.0,0.0
3,4024004,2026-06-03,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3500.0,0.0,0.0,3400.0,3070.0,...,100.0,6.208067e+08,6.208067e+08,8.132568e+07,1500.0,0.0,,0.0,0.0,0.0
4,4024005,2026-06-03,ABMM,ABM Investama Tbk.,XDMO1135000000A121------------,2390.0,2410.0,2410.0,2420.0,2260.0,...,100.0,2.753165e+09,2.753165e+09,4.138007e+08,411500.0,76500.0,,0.0,0.0,0.0
5,4024006,2026-06-03,ACES,Aspirasi Hidup Indonesia Tbk.,--MO1835UK6000E743------------,348.0,352.0,352.0,352.0,340.0,...,851200.0,1.712039e+10,1.712039e+10,6.846444e+09,8150000.0,14953500.0,,0.0,0.0,0.0
6,4024007,2026-06-03,ACRO,Samcro Hyosung Adilestari Tbk.,--UO2100000000E413------------,65.0,64.0,65.0,67.0,56.0,...,87800.0,3.469346e+09,7.492730e+08,7.299503e+08,3000.0,0.0,,0.0,0.0,0.0
7,4024008,2026-06-03,ACST,Acset Indonusa Tbk.,--U-4105000000J211-E---------X,83.0,0.0,0.0,82.0,78.0,...,34400.0,1.767516e+10,1.767516e+10,1.560717e+09,0.0,0.0,,0.0,0.0,0.0
8,4024009,2026-06-03,ADCP,Adhi Commuter Properti Tbk.,--UO2135000000H111M-----------,50.0,50.0,50.0,50.0,50.0,...,0.0,2.222222e+10,2.222222e+10,2.222222e+09,0.0,0.0,,0.0,0.0,0.0
9,4024010,2026-06-03,ADES,Akasha Wira International Tbk.,--UO1135000000D212------------,22225.0,22225.0,22300.0,22400.0,21000.0,...,300.0,5.898968e+08,5.898968e+08,5.031820e+07,6000.0,5100.0,,0.0,0.0,0.0


## 6. Load - Simpan ke Database SQLite

In [8]:
Session = sessionmaker(bind=engine)
session = Session()

# Hapus data lama dengan tanggal yang sama (idempotent)
date_val = df["date"].iloc[0] if len(df) > 0 else ""
existing = session.query(StockSummary).filter(StockSummary.date == date_val).count()
if existing > 0:
    print(f"Data tanggal {date_val} sudah ada ({existing} record). Menghapus data lama ...")
    session.query(StockSummary).filter(StockSummary.date == date_val).delete()
    session.commit()

# Bulk insert
db_records = []
for _, row in df.iterrows():
    db_records.append(StockSummary(
        id_stock_summary=int(row["id_stock_summary"]) if pd.notna(row["id_stock_summary"]) else 0,
        date=str(row["date"]),
        stock_code=str(row["stock_code"]),
        stock_name=str(row["stock_name"]),
        remarks=str(row.get("remarks", "")),
        previous=float(row["previous"]) if pd.notna(row["previous"]) else None,
        open_price=float(row["open_price"]) if pd.notna(row["open_price"]) else None,
        first_trade=float(row["first_trade"]) if pd.notna(row["first_trade"]) else None,
        high=float(row["high"]) if pd.notna(row["high"]) else None,
        low=float(row["low"]) if pd.notna(row["low"]) else None,
        close=float(row["close"]) if pd.notna(row["close"]) else None,
        change=float(row["change"]) if pd.notna(row["change"]) else None,
        volume=float(row["volume"]) if pd.notna(row["volume"]) else None,
        value=float(row["value"]) if pd.notna(row["value"]) else None,
        frequency=float(row["frequency"]) if pd.notna(row["frequency"]) else None,
        index_individual=float(row["index_individual"]) if pd.notna(row["index_individual"]) else None,
        offer=float(row["offer"]) if pd.notna(row["offer"]) else None,
        offer_volume=float(row["offer_volume"]) if pd.notna(row["offer_volume"]) else None,
        bid=float(row["bid"]) if pd.notna(row["bid"]) else None,
        bid_volume=float(row["bid_volume"]) if pd.notna(row["bid_volume"]) else None,
        listed_shares=float(row["listed_shares"]) if pd.notna(row["listed_shares"]) else None,
        tradeble_shares=float(row["tradeble_shares"]) if pd.notna(row["tradeble_shares"]) else None,
        weight_for_index=float(row["weight_for_index"]) if pd.notna(row["weight_for_index"]) else None,
        foreign_sell=float(row["foreign_sell"]) if pd.notna(row["foreign_sell"]) else None,
        foreign_buy=float(row["foreign_buy"]) if pd.notna(row["foreign_buy"]) else None,
        delisting_date=str(row.get("delisting_date", "")),
        non_regular_volume=float(row["non_regular_volume"]) if pd.notna(row["non_regular_volume"]) else None,
        non_regular_value=float(row["non_regular_value"]) if pd.notna(row["non_regular_value"]) else None,
        non_regular_frequency=float(row["non_regular_frequency"]) if pd.notna(row["non_regular_frequency"]) else None,
    ))

session.bulk_save_objects(db_records)
session.commit()

total = session.query(StockSummary).count()
print(f"[LOAD] Berhasil menyimpan {len(db_records)} record.")
print(f"[LOAD] Total record di database: {total}")
session.close()

[LOAD] Berhasil menyimpan 959 record.
[LOAD] Total record di database: 959


## 7. Verifikasi - Query Data dari Database

In [9]:
# Verifikasi: baca kembali data dari database
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM stock_summary LIMIT 10"))
    cols = result.keys()
    rows_db = result.fetchall()

df_verify = pd.DataFrame(rows_db, columns=cols)
print(f"Total record di database: {session.query(StockSummary).count()}")
print(f"\n10 baris pertama:")
df_verify[["stock_code", "stock_name", "close", "volume", "change"]]

Total record di database: 959

10 baris pertama:


,stock_code,stock_name,close,volume,change
0,AADI,Adaro Andalan Indonesia Tbk.,8000.0,23469400.0,-325.0
1,AALI,Astra Agro Lestari Tbk.,6650.0,4212500.0,-175.0
2,ABBA,Mahaka Media Tbk.,37.0,102900.0,0.0
3,ABDA,Asuransi Bina Dana Arta Tbk.,3120.0,8000.0,-380.0
4,ABMM,ABM Investama Tbk.,2320.0,1129000.0,-70.0
5,ACES,Aspirasi Hidup Indonesia Tbk.,348.0,40116300.0,0.0
6,ACRO,Samcro Hyosung Adilestari Tbk.,60.0,3860700.0,-5.0
7,ACST,Acset Indonusa Tbk.,79.0,939100.0,-4.0
8,ADCP,Adhi Commuter Properti Tbk.,50.0,22500.0,0.0
9,ADES,Akasha Wira International Tbk.,21950.0,100700.0,-275.0


In [10]:
# Statistik deskriptif
df.describe()

,id_stock_summary,previous,open_price,first_trade,high,low,close,change,volume,value,...,bid,bid_volume,listed_shares,tradeble_shares,weight_for_index,foreign_sell,foreign_buy,non_regular_volume,non_regular_value,non_regular_frequency
count,9.590000e+02,959.000000,959.000000,959.000000,959.000000,959.000000,959.000000,959.000000,9.590000e+02,9.590000e+02,...,959.000000,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,9.590000e+02,959.000000
mean,4.024480e+06,1278.291971,666.747654,666.528676,1153.471324,1074.186653,1241.554745,-36.737226,3.593146e+07,2.477591e+10,...,1064.060480,3.980887e+05,1.308886e+10,1.287635e+10,4.100741e+09,8.959305e+06,1.082736e+07,1.924439e+06,1.521644e+09,0.699687
std,2.769838e+02,6756.821623,1930.673485,1930.961738,6673.811092,6516.150943,6762.421305,133.280105,2.312600e+08,1.508646e+11,...,6266.217862,2.691906e+06,4.582396e+10,4.569680e+10,2.951355e+10,6.137694e+07,9.998123e+07,2.400532e+07,1.369107e+10,4.868362
min,4.024001e+06,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-1650.000000,0.000000e+00,0.000000e+00,...,0.000000,0.000000e+00,3.600000e+06,3.600000e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,4.024240e+06,100.000000,0.000000,0.000000,81.000000,71.000000,92.000000,-25.000000,1.042500e+05,4.378620e+07,...,61.000000,2.000000e+02,1.460281e+09,1.388310e+09,2.353932e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
50%,4.024480e+06,242.000000,108.000000,108.000000,216.000000,196.000000,234.000000,-7.000000,1.692900e+06,3.287438e+08,...,189.000000,6.600000e+03,3.571452e+09,3.534000e+09,7.245845e+08,6.430000e+04,2.700000e+04,0.000000e+00,0.000000e+00,0.000000
75%,4.024720e+06,802.500000,443.000000,440.000000,757.500000,695.000000,760.000000,-1.000000,1.311910e+07,3.061210e+09,...,675.000000,7.770000e+04,1.033562e+10,1.023714e+10,2.427630e+09,1.681600e+06,1.680250e+06,0.000000e+00,0.000000e+00,0.000000
max,4.024959e+06,190025.000000,22675.000000,22675.000000,194500.000000,191000.000000,191000.000000,975.000000,5.931078e+09,2.594530e+12,...,182225.000000,5.195740e+07,1.140573e+12,1.140573e+12,8.774430e+11,1.241292e+09,2.843005e+09,6.144034e+08,3.044101e+11,119.000000


## 8. Ekspor ke CSV (Opsional)

In [11]:
csv_path = "idx_stock_summary.csv"
df.to_csv(csv_path, index=False)
print(f"Data diekspor ke: {csv_path}")
print(f"Total: {len(df)} baris")

Data diekspor ke: idx_stock_summary.csv
Total: 959 baris
